# Topic 6 - Your First LLM API Call with Local Llama

This notebook uses raw HTTP and a local `llama-server` OpenAI-compatible endpoint. It intentionally uses Python's standard library only.

Start the server separately:

```text
llama-server -hf Qwen/Qwen3-4B-GGUF:Q4_K_M -ngl 0 -c 4096 --host 127.0.0.1 --port 8080
```

## Learning objectives

- identify endpoint, method, headers, authentication, and JSON body
- represent conversation history with `messages` and roles
- inspect content, usage, finish reason, headers, and status
- read provider documentation before coding
- triage `400`, `401`, and `429` at the HTTP layer

In [ ]:
import json
import os
import time
import urllib.error
import urllib.request
from pprint import pprint

BASE_URL = 'http://127.0.0.1:8080'
HEALTH_URL = f'{BASE_URL}/health'
MODELS_URL = f'{BASE_URL}/v1/models'
CHAT_URL = f'{BASE_URL}/v1/chat/completions'
MODEL = os.getenv('LLAMA_MODEL', 'Qwen/Qwen3-4B-GGUF:Q4_K_M')
print(CHAT_URL)

## 1. Raw HTTP anatomy

A request is `POST` plus an endpoint, headers, authentication when required, and a JSON body. The local server normally needs no API key, but many cloud providers require `Authorization: Bearer <key>`.

This local endpoint follows the OpenAI-compatible shape: `POST /v1/chat/completions`.

In [ ]:
headers = {'Content-Type': 'application/json'}
body = {
    'model': MODEL,
    'messages': [
        {'role': 'system', 'content': 'Answer concisely.'},
        {'role': 'user', 'content': 'Explain what an API endpoint is in one sentence.'},
    ],
    'temperature': 0.2,
    'max_tokens': 200,
}

print('METHOD: POST')
print('ENDPOINT:', CHAT_URL)
print('HEADERS:', headers)
print('BODY:', json.dumps(body, indent=2))
assert body['messages'][0]['role'] == 'system'
assert body['messages'][1]['role'] == 'user'

In [ ]:
def request_json(url, method='GET', body=None, headers=None, timeout=30):
    request_headers = {'Accept': 'application/json', **(headers or {})}
    data = None if body is None else json.dumps(body).encode('utf-8')
    request = urllib.request.Request(url, data=data, headers=request_headers, method=method)
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw = response.read().decode('utf-8')
            return {'status': response.status, 'headers': dict(response.headers), 'body': json.loads(raw), 'elapsed': time.perf_counter() - started}
    except urllib.error.HTTPError as error:
        raw = error.read().decode('utf-8', errors='replace')
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            parsed = {'raw': raw}
        return {'status': error.code, 'headers': dict(error.headers), 'body': parsed, 'elapsed': time.perf_counter() - started}
    except urllib.error.URLError as error:
        return {'status': None, 'headers': {}, 'body': {'network_error': str(error.reason)}, 'elapsed': time.perf_counter() - started}

## 2. Check the local API

Health and model-list requests are useful before debugging generation. If these fail, the issue is local server availability rather than prompt content.

In [ ]:
health_result = request_json(HEALTH_URL)
models_result = request_json(MODELS_URL)
print('Health:', health_result['status'], json.dumps(health_result['body'], indent=2))
print('Models:', models_result['status'], json.dumps(models_result['body'], indent=2))

## 3. Make the first raw HTTP LLM call

No SDK is involved. The result retains status, response headers, parsed JSON, and elapsed time.

In [ ]:
result = request_json(CHAT_URL, method='POST', body=body, headers={'Content-Type': 'application/json'}, timeout=180)
print('HTTP status:', result['status'])
print('Elapsed seconds:', round(result['elapsed'], 2))
print('Response headers:')
for key, value in result['headers'].items():
    print(f'  {key}: {value}')
print('Response body:')
print(json.dumps(result['body'], indent=2, ensure_ascii=False))

## 4. Read every response field

A `200` is only HTTP success. Inspect application fields such as `choices`, `message.content`, `finish_reason`, and `usage`. Some reasoning models may expose reasoning separately or consume part of the output budget before final content.

In [ ]:
response_body = result['body']
print('Top-level keys:', list(response_body.keys()) if isinstance(response_body, dict) else type(response_body).__name__)
choices = response_body.get('choices', []) if isinstance(response_body, dict) else []
if choices:
    choice = choices[0]
    message = choice.get('message', {})
    print('Content:', message.get('content'))
    print('Reasoning content:', message.get('reasoning_content'))
    print('Finish reason:', choice.get('finish_reason'))
else:
    print('No choices; inspect the complete error or response body above.')
print('Usage:')
pprint(response_body.get('usage') if isinstance(response_body, dict) else None)

In [ ]:
messages_example = [
    {'role': 'system', 'content': 'Use technical language.'},
    {'role': 'user', 'content': 'What is HTTP 429?'},
    {'role': 'assistant', 'content': 'It indicates rate limiting.'},
    {'role': 'user', 'content': 'What should I inspect next?'},
]
for index, message in enumerate(messages_example):
    print(index, message['role'], '->', message['content'])
assert [message['role'] for message in messages_example] == ['system', 'user', 'assistant', 'user']

## 5. Error triage: `400`, `401`, and `429`

A `400` means inspect the request body, endpoint, field names, types, and values. A `401` means inspect authentication headers and credentials. A `429` means inspect rate, quota, spend limits, and retry guidance such as `Retry-After`.

The local server normally has no authentication, so this notebook does not pretend to produce a real local `401`. The `400` exercise below is safe and real; use an authenticated cloud provider for a real `401` test.

In [ ]:
bad_body = {**body, 'messages': 'this should be a list'}
bad_result = request_json(CHAT_URL, method='POST', body=bad_body, headers={'Content-Type': 'application/json'}, timeout=30)
print('HTTP status:', bad_result['status'])
print(json.dumps(bad_result['body'], indent=2))
print('Diagnosis:', 'request problem' if bad_result['status'] == 400 else 'read the actual response body')

In [ ]:
def classify_http_status(status):
    if status == 400:
        return 'request problem'
    if status == 401:
        return 'authentication problem'
    if status == 429:
        return 'rate/quota problem'
    if status is not None and 200 <= status < 300:
        return 'HTTP success; inspect application fields'
    return 'read the provider response and documentation'

for status in [200, 400, 401, 429, 500]:
    print(status, '->', classify_http_status(status))
assert classify_http_status(400) == 'request problem'
assert classify_http_status(401) == 'authentication problem'
assert classify_http_status(429) == 'rate/quota problem'

## 6. Documentation-reading checklist

Before adapting this notebook to a provider, verify:

1. exact endpoint and HTTP method
2. required headers and authentication format
3. required JSON fields and accepted roles
4. response paths for content, usage, and stop/finish reason
5. error schema, request IDs, and retry headers

A local OpenAI-compatible endpoint shares a useful shape with cloud APIs, but it does not guarantee identical models, limits, authentication, or response behavior.

## Exercises

- change the messages and compare the complete response
- change `max_tokens` and compare `finish_reason` and usage
- try a structured JSON response format if your server supports it
- capture a real response and write assertions for its fields
- use environment variables for a cloud endpoint, key, and model
- explain why a local server is useful for HTTP learning but not identical to a hosted provider

Do not disable TLS verification or spam a service to force a rate limit.

## Topic 6 completion checklist

- [ ] I can identify endpoint, method, headers, authentication, and JSON body.
- [ ] I can explain the `messages` array and roles.
- [ ] I can locate content, usage, and finish reason.
- [ ] I understand that HTTP `200` is different from a model stop reason.
- [ ] I can triage `400`, `401`, and `429` without guessing.
- [ ] I can make the call without an SDK.